# Ejercicio: asistente de documentos con NVIDIA NIM y Gradio

**Tiempo:** 40–45 minutos  
**Grupos:** 3–4 personas

## Contexto

En los notebooks anteriores construimos chatbots que responden preguntas
generales. En este ejercicio construiremos un asistente que responda preguntas
sobre un documento específico: **“Attention Is All You Need”**
(Vaswani et al., 2017).

Inyectar un documento completo en el contexto permite construir un primer
prototipo de *document question answering*. Esto ayuda a entender la motivación
de **RAG**, aunque todavía no es RAG: aquí no existe recuperación de fragmentos.

## Objetivo

Construir una aplicación Gradio donde el usuario pueda:

1. Cargar o leer el PDF del paper.
2. Hacer preguntas sobre su contenido.
3. Obtener respuestas basadas exclusivamente en el documento.

## Lo que van a aprender

- Extracción de texto de PDF con `pypdf`.
- Uso de contexto externo en un `system` prompt.
- Consumo de NVIDIA NIM mediante una API compatible con OpenAI.
- Streaming e historial con Gradio.
- Limitaciones de este enfoque y motivación de RAG.

## Paso 0: instalación y configuración

### 0.1 Instala las dependencias

- `pypdf`: extracción del texto del PDF.
- `gradio`: interfaz web.
- `openai`: cliente compatible con NVIDIA NIM.
- `python-dotenv`: lectura de variables de entorno.

```bash
pip install -q --upgrade pypdf gradio openai python-dotenv
```

In [1]:
!pip install -q --upgrade pypdf gradio openai python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 949.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.7/401.7 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 82.4 MB/s eta 0:00:00


### 0.2 Descarga el paper

Descarga el PDF abierto desde:

```text
https://arxiv.org/pdf/1706.03762
```

Guárdalo junto al notebook como `attention_is_all_you_need.pdf`.

### 0.3 Configura tus credenciales

Crea un archivo `.env` con tu API key de NVIDIA:

```text
NVIDIA_API_KEY="tu_key_aqui"
```

Puedes crearla en [NVIDIA Build](https://build.nvidia.com/settings/api-keys).

## Paso 1: extracción de texto del PDF

Lee el PDF y extrae su contenido como texto plano.

**Instrucciones:**

1. Importa `PdfReader` desde `pypdf`.
2. Crea `extract_text_from_pdf(pdf_path)`.
3. Abre el PDF e itera sobre `reader.pages`.
4. Concatena el resultado de `page.extract_text()`; contempla páginas sin texto.
5. Retorna el texto completo.
6. Prueba la función e imprime los primeros 500 caracteres.

> **Pista:** usa `page.extract_text() or ""` para evitar valores `None`.

In [5]:
from google.colab import drive
from getpass import getpass
import os

# 1. Montar Drive
drive.mount("/content/drive")

# 2. Ir a la carpeta del ejercicio (ajusta la ruta a la tuya)
CARPETA = "/content/drive/MyDrive/AI/Clase 9"
os.chdir(CARPETA)
print("Carpeta actual:", os.getcwd())
print("Archivos:", os.listdir("."))

# 3. Crear el .env pidiendo la key de forma oculta (solo la primera vez)
if not os.path.exists(".env"):
    key = getpass("Pega tu NVIDIA_API_KEY: ")
    with open(".env", "w") as f:
        f.write(f'NVIDIA_API_KEY="{key}"\n')
    print(".env creado")
else:
    print(".env ya existe")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Carpeta actual: /content/drive/MyDrive/AI/Clase 9
Archivos: ['01_local_chatbot_lab.ipynb', '02_local_remote_hf_chatbot_lab.ipynb', '03_nvidia_chatbot_lab.ipynb', '04_nvidia_gradio_lab.ipynb', 'attention_is_all_you_need.pdf', '05_ejercicio_nvidia_documentos.ipynb']
Pega tu NVIDIA_API_KEY: ··········
.env creado


In [6]:
from pypdf import PdfReader

def extract_text_from_pdf(pdf_path):
    """Lee un PDF y devuelve todo su texto como un solo string."""
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        text += (page.extract_text() or "") + "\n"  # "or ''" evita None en páginas sin texto
    return text

PDF_PATH = "attention_is_all_you_need.pdf"
document_text = extract_text_from_pdf(PDF_PATH)

print(f"Páginas: {len(PdfReader(PDF_PATH).pages)}")
print(f"Caracteres: {len(document_text)}")
print(document_text[:500])

Páginas: 15
Caracteres: 39600
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz 


## Paso 2: inicialización del cliente de NVIDIA NIM

**Instrucciones:**

1. Importa `os`, `load_dotenv` y `OpenAI`.
2. Carga las variables con `load_dotenv()`.
3. Verifica que `NVIDIA_API_KEY` esté disponible sin imprimirla.
4. Inicializa `OpenAI` con:
   - `base_url="https://integrate.api.nvidia.com/v1"`
   - `api_key=os.getenv("NVIDIA_API_KEY")`
5. Define:

```python
MODELO = "nvidia/nemotron-3-super-120b-a12b"
```

In [7]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# Carga el .env de la carpeta actual (la de Drive donde hiciste os.chdir)
load_dotenv(".env", override=True)

# Verifica que la key exista sin imprimirla
api_key = os.getenv("NVIDIA_API_KEY")
if not api_key:
    raise ValueError("No se encontró NVIDIA_API_KEY. Revisa que el .env esté en la carpeta actual.")
print("NVIDIA_API_KEY cargada correctamente")

# Cliente OpenAI apuntando a NVIDIA NIM
client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=api_key,
)

MODELO = "nvidia/nemotron-3-super-120b-a12b"

NVIDIA_API_KEY cargada correctamente


## Paso 3: función de chat con el documento

Construye un `system` prompt que incluya el texto del paper y obligue al modelo
a fundamentar sus respuestas en él.

**Instrucciones:**

1. Crea `build_system_prompt(document_text)` para:
   - definir al asistente como experto en el paper;
   - incluir el documento entre delimitadores claros;
   - exigir respuestas basadas únicamente en el documento;
   - indicar que responda “No encuentro esa información en el documento” cuando corresponda.

2. Crea `history_to_messages(history)` para convertir el historial de Gradio a
   mensajes con roles `user` y `assistant`.

3. Crea `chat_con_documento(message, history, document_text)` para:
   - iniciar `messages` con el `system` prompt;
   - agregar el historial y el mensaje actual;
   - llamar `client.chat.completions.create(..., stream=True)`;
   - acumular `chunk.choices[0].delta.content`;
   - usar `yield` para entregar la respuesta progresivamente.

> **Pista:** utiliza `temperature=1.0` y `top_p=0.95`, valores recomendados para
> Nemotron 3 Super.

In [8]:
def build_system_prompt(document_text):
    """System prompt que fundamenta al modelo en el documento."""
    return f"""Eres un asistente experto en el paper "Attention Is All You Need" (Vaswani et al., 2017).

Responde las preguntas del usuario basándote ÚNICAMENTE en el documento que aparece entre los delimitadores.
Si la respuesta no está en el documento, responde exactamente: "No encuentro esa información en el documento."
No uses conocimiento externo. Responde en el mismo idioma en que te pregunte el usuario.

<documento>
{document_text}
</documento>"""


def history_to_messages(history):
    """Convierte el historial de Gradio a mensajes con roles user/assistant.
    Soporta formato 'messages' (lista de dicts) y 'tuples' (lista de pares)."""
    messages = []
    for item in history:
        if isinstance(item, dict):  # formato messages
            content = item["content"]
            if isinstance(content, list):  # contenido multimodal: nos quedamos con el texto
                content = " ".join(
                    part.get("text", "") for part in content if isinstance(part, dict)
                )
            messages.append({"role": item["role"], "content": content})
        else:  # formato tuples: (usuario, asistente)
            user_msg, assistant_msg = item
            if user_msg:
                messages.append({"role": "user", "content": user_msg})
            if assistant_msg:
                messages.append({"role": "assistant", "content": assistant_msg})
    return messages


def chat_con_documento(message, history, document_text):
    """Genera la respuesta en streaming, fundamentada en el documento."""
    messages = [{"role": "system", "content": build_system_prompt(document_text)}]
    messages += history_to_messages(history)
    messages.append({"role": "user", "content": message})

    stream = client.chat.completions.create(
        model=MODELO,
        messages=messages,
        temperature=1.0,
        top_p=0.95,
        max_tokens=2048,
        stream=True,
    )

    respuesta = ""
    for chunk in stream:
        if not chunk.choices:
            continue
        delta = chunk.choices[0].delta.content
        if delta:
            respuesta += delta
            yield respuesta

In [9]:
for parcial in chat_con_documento("¿Cuántas capas tiene el encoder del modelo base?", [], document_text):
    pass
print(parcial)

6


## Paso 4: interfaz Gradio

Construye la interfaz con `gr.ChatInterface`.

**Instrucciones:**

1. Extrae el texto del PDF con la función del paso 1.
2. Muestra cuántas páginas y caracteres se extrajeron.
3. Configura `gr.ChatInterface` con:
   - `fn=chat_con_documento`;
   - un título y una descripción;
   - un `gr.Textbox` oculto dentro de `additional_inputs` para enviar el documento;
   - al menos tres preguntas de ejemplo.

```python
gr.Textbox(value=document_text, visible=False)
```

4. Lanza la interfaz:

```python
demo.launch(server_name="0.0.0.0", server_port=8080, show_error=True)
```

In [11]:
import gradio as gr
from pypdf import PdfReader

# 1-2. Extraer el texto y mostrar cuántas páginas y caracteres se leyeron
PDF_PATH = "attention_is_all_you_need.pdf"
document_text = extract_text_from_pdf(PDF_PATH)
n_paginas = len(PdfReader(PDF_PATH).pages)
print(f"Páginas: {n_paginas} | Caracteres extraídos: {len(document_text):,}")

# 3. Preguntas de ejemplo (con additional_inputs, cada ejemplo lleva también el valor del texto oculto)
preguntas = [
    "¿Cuál es la arquitectura principal propuesta en el paper?",
    "¿Qué es el mecanismo de atención?",
    "¿Cuántas capas tiene el encoder del modelo base?",
    "¿Cuál es el resultado del modelo en la tarea WMT 2014 English-to-German?",
]

demo = gr.ChatInterface(
    fn=chat_con_documento,
    title="Asistente del paper: Attention Is All You Need",
    description="Haz preguntas sobre el paper de Vaswani et al. (2017). "
                "Las respuestas se basan únicamente en el documento.",
    additional_inputs=[gr.Textbox(value=document_text, visible=False)],
    examples=[[q, document_text] for q in preguntas],
)

# 4. Lanzar la interfaz
demo.launch(show_error=True)

Páginas: 15 | Caracteres extraídos: 39,600
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://80d0fbe98a1065cc27.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Paso 5: Prueba y reflexión

Una vez que la interfaz esté funcionando, prueba estas preguntas:

Aquí están las 6 preguntas del Paso 5, cada una con la respuesta tal como me la pasaste.

**1. ¿Cuál es la arquitectura principal propuesta en el paper?**

La arquitectura principal propuesta en el artículo es el **Transformer**, una red de transduction de secuencias que se basa únicamente en mecanismos de atención (multi‑head self‑attention) y elimina completamente las capas recurrentes y convolucionales. El Transformer consta de un codificador y un decodificador apilados, cada uno formado por \(N=6\) capas idénticas que contienen:

- una sub‑capa de atención multi‑cabezal,
- una sub‑capa de red feed‑forward punto‑a‑punto (dos transformaciones lineales con ReLU entre ellas),
- conexiones residuales y normalización de capa alrededor de cada sub‑capa,
- y, en el decodificador, una sub‑capa adicional de atención sobre la salida del codificador.

Además, se añaden codificaciones posicionales (seno‑coseno) para proporcionar información del orden de las secuencias. Esta arquitectura permite una mayor paralelización y reduce el número de operaciones secuenciales a \(O(1)\) por capa, mejorando tanto la eficiencia de entrenamiento como la calidad de traducción.

**2. ¿Qué es el mecanismo de atención?**

Según el documento, el mecanismo de atención se describe como una función que asigna una **consulta** (query) y un conjunto de pares **clave‑valor** (key‑value) a una salida. La salida se calcula como una suma ponderada de los valores, donde el peso asignado a cada valor proviene de una función de compatibilidad entre la consulta y la correspondiente clave. En el trabajo se utiliza la variante llamada *Scaled Dot‑Product Attention*: se calcula el producto punto entre la consulta y cada clave, se escala dividiendo por √dk, se aplica un softmax para obtener los pesos y finalmente se multiplica esos pesos por los valores. Esta operación permite que el modelo modele dependencias entre posiciones sin importar su distancia en la secuencia.

**3. ¿Cuántas capas tiene el encoder del modelo base?**

El encoder del modelo base está compuesto por \(N = 6\) capas idénticas.

**4. ¿Quiénes son los autores del paper?**

Los autores del artículo son: Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser e Illia Polosukhin.

**5. ¿Cuál es el resultado del modelo en la tarea WMT 2014 English-to-German?**

28.4 BLEU (en la tarea WMT 2014 English‑to‑German).

**6. ¿Qué es GPT-4?**

No encuentro esa información en el documento.

¿Usa su conocimiento general o respeta la instrucción de ceñirse al documento?

Respeta la instrucción: ante “¿Qué es GPT-4?” respondió “No encuentro esa información en el documento.” en lugar de recurrir a su conocimiento general.

## Paso 6 (adicional): mejora el sistema

Si terminan antes, implementen una mejora:

**A) Medición del contexto**  
Reporta la cantidad de caracteres o estima los tokens del documento. Explica
por qué caracteres y tokens no son equivalentes.

**B) Subida dinámica de PDF**  
Agrega `gr.File` para cargar cualquier PDF y extrae el texto dentro del flujo de
la aplicación.

**C) Evidencia y páginas**  
Conserva separadores de página al extraer el PDF y exige que cada respuesta cite
la página que contiene la evidencia.

**D) Primer paso hacia RAG**  
Divide el documento en fragmentos y selecciona los más relevantes mediante una
búsqueda simple por palabras antes de llamar al modelo.

In [12]:
# 6A
# 1. Caracteres y palabras
n_chars = len(document_text)
n_words = len(document_text.split())
print(f"Caracteres: {n_chars:,}")
print(f"Palabras:   {n_words:,}")

# 2. Estimación rápida de tokens (regla práctica: ~4 caracteres por token en inglés)
print(f"Tokens estimados (chars/4): {n_chars // 4:,}")

# 3. Conteo real con el tokenizador del servidor: el campo usage de la respuesta
#    (incluye el system prompt completo con el documento)
resp = client.chat.completions.create(
    model=MODELO,
    messages=[
        {"role": "system", "content": build_system_prompt(document_text)},
        {"role": "user", "content": "Hola"},
    ],
    max_tokens=1,
)
tokens_reales = resp.usage.prompt_tokens
print(f"Tokens reales del prompt:   {tokens_reales:,}")
print(f"Caracteres por token real:  {n_chars / tokens_reales:.2f}")

Caracteres: 39,600
Palabras:   6,111
Tokens estimados (chars/4): 9,900
Tokens reales del prompt:   10,996
Caracteres por token real:  3.60


¿Por qué caracteres y tokens no son equivalentes?
Un token es una unidad del vocabulario del modelo (una palabra completa, un fragmento de palabra o un símbolo), así que su longitud varía: las palabras comunes en inglés son un solo token, mientras que fórmulas, símbolos matemáticos, nombres propios, referencias y otros idiomas se parten en más tokens por cada carácter.

In [17]:
import os
import gradio as gr
from pypdf import PdfReader

# ---- Lo que faltaba del 6C ----
def extract_text_with_pages(pdf_path):
    reader = PdfReader(pdf_path)
    partes = []
    for i, page in enumerate(reader.pages, start=1):
        texto = page.extract_text() or ""
        partes.append(f"[[PÁGINA {i}]]\n{texto}")
    return "\n\n".join(partes)


def build_system_prompt(document_text):
    return f"""Eres un asistente experto en el documento que aparece entre los delimitadores.

Reglas:
1. Responde ÚNICAMENTE con información del documento. No uses conocimiento externo.
2. El documento está dividido con marcadores [[PÁGINA N]]. Al final de cada afirmación importante, cita la página que la respalda con el formato (p. N).
3. Si la respuesta no está en el documento, responde exactamente: "No encuentro esa información en el documento." y no cites ninguna página.
4. Responde en el mismo idioma en que te pregunte el usuario.

<documento>
{document_text}
</documento>"""


# ---- Paso 6B ----
DEFAULT_PDF = "dependencia_total_ejercicio.pdf"
_cache = {"ruta": None, "mtime": None, "texto": ""}

def cargar_documento(archivo):
    """Devuelve (texto, nombre). Si no hay archivo, usa el PDF por defecto."""
    ruta = archivo if isinstance(archivo, str) else getattr(archivo, "name", None)
    if not ruta:
        ruta = DEFAULT_PDF
    if not ruta.lower().endswith(".pdf"):
        raise ValueError("El archivo debe ser un PDF.")

    mtime = os.path.getmtime(ruta)
    if _cache["ruta"] != ruta or _cache["mtime"] != mtime:
        _cache.update(ruta=ruta, mtime=mtime, texto=extract_text_with_pages(ruta))
    return _cache["texto"], os.path.basename(ruta)


def chat_con_pdf_dinamico(message, history, archivo):
    try:
        texto, nombre = cargar_documento(archivo)
    except Exception as e:
        yield f"No pude leer el PDF: {e}"
        return

    if len(texto.replace("[[PÁGINA", "").strip()) < 200:
        yield "El PDF no contiene texto extraíble (¿es un documento escaneado?)."
        return

    yield from chat_con_documento(message, history, texto)


try:
    demo.close()
except Exception:
    pass

demo = gr.ChatInterface(
    fn=chat_con_pdf_dinamico,
    title="Asistente de documentos PDF",
    description="Sube cualquier PDF y haz preguntas sobre él. "
                "Si no subes ninguno, se usa el paper 'Attention Is All You Need'. "
                "Las respuestas se basan únicamente en el documento.",
    additional_inputs=[gr.File(label="Sube un PDF (opcional)", file_types=[".pdf"])],
    examples=[
        ["¿De qué trata este documento?", None],
        ["¿Cuántas capas tiene el encoder del modelo base?", None],
        ["¿Qué es GPT-4?", None],
    ],
)

demo.launch(show_error=True)

Closing server running on port: 7860
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8e2546b167b3c070b5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [20]:
#6C

import gradio as gr
from pypdf import PdfReader

# 1. Extracción con separadores de página
def extract_text_with_pages(pdf_path):
    reader = PdfReader(pdf_path)
    partes = []
    for i, page in enumerate(reader.pages, start=1):
        texto = page.extract_text() or ""
        partes.append(f"[[PÁGINA {i}]]\n{texto}")
    return "\n\n".join(partes)

PDF_PATH = "dependencia_total_ejercicio.pdf"
document_text = extract_text_with_pages(PDF_PATH)
print(document_text[:300])

# 2. System prompt que exige citar la página de evidencia
def build_system_prompt(document_text):
    return f"""Eres un asistente experto en el documento que aparece entre los delimitadores.

Reglas:
1. Responde ÚNICAMENTE con información del documento. No uses conocimiento externo.
2. El documento está dividido con marcadores [[PÁGINA N]]. Al final de cada afirmación importante, cita la página que la respalda con el formato (p. N).
3. Si la respuesta no está en el documento, responde exactamente: "No encuentro esa información en el documento." y no cites ninguna página.
4. Responde en el mismo idioma en que te pregunte el usuario.

<documento>
{document_text}
</documento>"""

# 3. Interfaz con el texto paginado
try:
    demo.close()
except Exception:
    pass

preguntas = [
    "¿Cuántas capas tiene el encoder del modelo base?",
    "¿Cuál es el resultado del modelo en la tarea WMT 2014 English-to-German?",
    "¿Qué es el mecanismo de atención?",
    "¿Qué es GPT-4?",
]

demo = gr.ChatInterface(
    fn=chat_con_documento,
    title="Asistente del paper: Attention Is All You Need (con citas)",
    description="Respuestas basadas únicamente en el documento, citando la página de evidencia.",
    additional_inputs=[gr.Textbox(value=document_text, visible=False)],
    examples=[[q, document_text] for q in preguntas],
)

demo.launch(show_error=True)

[[PÁGINA 1]]
Dependencia
 
Total
 
 
Intro:
 
 
●
 
Has
 
intentando
 
hacer
 
las
 
cosas
 
con
 
tus
 
fuerzas
 
y
 
has
 
fallado?
 
●
 
Has
 
tenido
 
una
 
lucha
 
en
 
tus
 
pensamientos
 
sobre
 
que
 
hacer
 
o
 
no
 
hacer?
 
●
 
Has
 
perdido
 
el
 
dominio
 
sobre
 
tus
 
emociones?
 
 
●
Closing server running on port: 7860
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://557b82b5c9397785f0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [24]:
import os

# 1. Apuntar al nuevo PDF por defecto
DEFAULT_PDF = "dependencia_total_ejercicio.pdf"
assert os.path.exists(DEFAULT_PDF), f"No encuentro {DEFAULT_PDF} en {os.getcwd()}"

# 2. Idioma de las palabras clave: cámbialo a "INGLÉS" si el PDF está en inglés
IDIOMA_DOC = "ESPAÑOL"

def palabras_clave(pregunta):
    try:
        r = client.chat.completions.create(
            model=MODELO,
            messages=[{"role": "user", "content":
                f"Extrae 6 a 10 palabras clave en {IDIOMA_DOC} para buscar la respuesta a esta pregunta "
                "en el documento. Responde solo con las palabras separadas por espacios, "
                f"sin explicación.\n\nPregunta: {pregunta}"}],
            temperature=0.2, max_tokens=60,
        )
        extra = r.choices[0].message.content or ""
    except Exception:
        extra = ""
    return f"{pregunta} {extra}"

# 3. Verificación rápida: cuántas páginas y fragmentos genera
texto, nombre = cargar_documento(None)
frags = dividir_en_fragmentos(texto)
print(f"{nombre}: {len(texto):,} caracteres, {len(frags)} fragmentos")

dependencia_total_ejercicio.pdf: 4,438 caracteres, 5 fragmentos


In [25]:
#6D

import re
import math
import gradio as gr
from collections import Counter

# ---------- 1. Fragmentación (conserva el número de página) ----------
def dividir_en_fragmentos(texto_con_paginas, tam=1200, solape=200):
    partes = re.split(r"\[\[PÁGINA (\d+)\]\]", texto_con_paginas)
    # partes = ['', '1', texto_p1, '2', texto_p2, ...]
    fragmentos = []
    for i in range(1, len(partes), 2):
        pagina = int(partes[i])
        contenido = partes[i + 1].strip()
        inicio = 0
        while inicio < len(contenido):
            trozo = contenido[inicio:inicio + tam].strip()
            if trozo:
                fragmentos.append({"pagina": pagina, "texto": trozo})
            inicio += tam - solape
    return fragmentos

# ---------- 2. Búsqueda simple por palabras (TF-IDF básico) ----------
STOPWORDS = set("""el la los las un una unos unas de del al y o en que es son por para con se su sus lo
como qué cuál cuáles cuántas cuántos the a an of to in and or is are for with on at by this that it as""".split())

def tokenizar(texto):
    return [t for t in re.findall(r"\w+", texto.lower()) if t not in STOPWORDS and len(t) > 1]

def buscar_fragmentos(consulta, fragmentos, k=4):
    terminos = set(tokenizar(consulta))
    n = len(fragmentos)
    tokens_frag = [Counter(tokenizar(f["texto"])) for f in fragmentos]
    df = Counter()
    for c in tokens_frag:
        for t in c:
            df[t] += 1

    puntajes = []
    for i, c in enumerate(tokens_frag):
        score = sum(c[t] * math.log((n + 1) / (df[t] + 1) + 1) for t in terminos if t in c)
        puntajes.append((score, i))
    puntajes.sort(reverse=True)
    return [fragmentos[i] for s, i in puntajes[:k] if s > 0]

# ---------- 3. Palabras clave en el idioma del documento ----------
def palabras_clave(pregunta):
    try:
        r = client.chat.completions.create(
            model=MODELO,
            messages=[{"role": "user", "content":
                "Extrae 6 a 10 palabras clave en INGLÉS para buscar la respuesta a esta pregunta "
                "en un artículo científico. Responde solo con las palabras separadas por espacios, "
                f"sin explicación.\n\nPregunta: {pregunta}"}],
            temperature=0.2, max_tokens=60,
        )
        extra = r.choices[0].message.content or ""
    except Exception:
        extra = ""
    return f"{pregunta} {extra}"

# ---------- 4. Chat con recuperación ----------
_cache_frag = {"ruta": None, "mtime": None, "frags": []}

def chat_rag(message, history, archivo):
    try:
        texto, nombre = cargar_documento(archivo)
    except Exception as e:
        yield f"No pude leer el PDF: {e}"
        return

    ruta = _cache["ruta"]
    if _cache_frag["ruta"] != ruta or _cache_frag["mtime"] != _cache["mtime"]:
        _cache_frag.update(ruta=ruta, mtime=_cache["mtime"], frags=dividir_en_fragmentos(texto))

    consulta = palabras_clave(message)
    elegidos = buscar_fragmentos(consulta, _cache_frag["frags"], k=4)

    if not elegidos:
        yield "No encuentro esa información en el documento."
        return

    contexto = "\n\n".join(f"[[PÁGINA {f['pagina']}]]\n{f['texto']}" for f in elegidos)
    messages = [{"role": "system", "content": build_system_prompt(contexto)}]
    messages += history_to_messages(history)
    messages.append({"role": "user", "content": message})

    stream = client.chat.completions.create(
        model=MODELO, messages=messages,
        temperature=1.0, top_p=0.95, max_tokens=2048, stream=True,
    )

    respuesta = ""
    for chunk in stream:
        if not chunk.choices:
            continue
        delta = chunk.choices[0].delta.content
        if delta:
            respuesta += delta
            yield respuesta

    paginas = sorted({f["pagina"] for f in elegidos})
    yield respuesta + f"\n\n_Fragmentos recuperados de las páginas: {', '.join(map(str, paginas))}_"

# ---------- 5. Interfaz ----------
try:
    demo.close()
except Exception:
    pass

demo = gr.ChatInterface(
    fn=chat_rag,
    title="Asistente de documentos con recuperación (mini-RAG)",
    description="Divide el PDF en fragmentos, recupera los más relevantes y responde solo con ellos.",
    additional_inputs=[gr.File(label="Sube un PDF (opcional)", file_types=[".pdf"])],
    examples=[
        ["¿Que es la obediencia?", None],
        ["¿Que personajes biblicos se mencionan?", None],
        ["¿Cuales son los puntos principales", None],
    ],
)
demo.launch(show_error=True)

Closing server running on port: 7860
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c7ed0a0fdddfd966fe.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Reflexión final

Discutan en grupo:

1. ¿Cuál es la limitación de enviar siempre el documento completo?
Cada pregunta consume todo el contexto del paper (unos 40.000 caracteres), lo que aumenta costo y latencia, y no escala a documentos largos o a muchos documentos que superen la ventana de contexto del modelo.

2. ¿Por qué este ejercicio todavía no implementa RAG?
Porque no hay recuperación: no se divide el documento en fragmentos ni se seleccionan los más relevantes para cada pregunta, simplemente se inyecta todo el texto en el system prompt.

3. ¿Cómo comprobarían que la respuesta proviene del documento y no del conocimiento previo del modelo?
Exigiendo que cada respuesta cite la página o el fragmento exacto que la respalda (Paso 6C) y probando con preguntas que el documento no contiene, como la de GPT-4 o la bandera de Colombia (pregunta personal que se realizo), donde el modelo respondió correctamente “No encuentro esa información en el documento”.

4. ¿Qué riesgos aparecen al insertar directamente documentos de usuarios en el prompt?
Un documento malicioso puede incluir instrucciones ocultas (prompt injection) que alteren el comportamiento del asistente o filtren información, además del riesgo de exponer datos sensibles a un servicio externo y de saturar el contexto con archivos enormes.

## Recursos

- [pypdf](https://pypdf.readthedocs.io)
- [Gradio](https://www.gradio.app/docs)
- [NVIDIA Build](https://build.nvidia.com)
- [Nemotron 3 Super](https://build.nvidia.com/nvidia/nemotron-3-super-120b-a12b)
- [Paper original](https://arxiv.org/abs/1706.03762)